In [ ]:
!pip install openpyxl
!pip install imblearn
!pip install opencv-python # Installs the OpenCV library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import RFE, mutual_info_classif

import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision.models as models
import torch.optim as optim
from torchvision import transforms

from imblearn.over_sampling import SMOTE

import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Define dataset path
pcos_extended_path = "/content/drive/MyDrive/pcos_detection/PCOS_extended_dataset.csv"  # Update with actual path

# Load the new dataset (PCOS Extended Data)
df_pcos_extended = pd.read_csv(pcos_extended_path)

# Display dataset information
print("✅ PCOS Extended Dataset Loaded!")
print("📊 Dataset Shape:", df_pcos_extended.shape)
print(df_pcos_extended.head())


✅ PCOS Extended Dataset Loaded!
📊 Dataset Shape: (2000, 44)
   Sl. No  Patient File No.  PCOS (Y/N)   Age (yrs)  Weight (Kg)  Height(Cm)   \
0     193               193           0          30    69.979147   167.708055   
1     360               360           0          36    63.711688   154.055877   
2      10                10           0          36    51.848631   149.059804   
3     278               278           1          29    66.893988   148.628036   
4      71                71           0          33    52.536198   150.767409   

         BMI  Blood Group  Pulse rate(bpm)   RR (breaths/min)  ...  \
0  23.185569           12                72                22  ...   
1  25.441392           13                70                18  ...   
2  23.928264           15                80                20  ...   
3  27.894935           15                72                18  ...   
4  23.079564           13                72                18  ...   

   Pimples(Y/N)  Fast food (Y/N)

In [ ]:
# Check missing values before fixing
print("Missing Values Before Fixing:")
print(df_pcos_extended.isnull().sum())

# Handle missing values:
for col in df_pcos_extended.columns:
    if df_pcos_extended[col].dtype == "object":  # For categorical columns
        df_pcos_extended[col].fillna(df_pcos_extended[col].mode()[0], inplace=True)
    else:  # For numerical columns
        df_pcos_extended[col].fillna(df_pcos_extended[col].median(), inplace=True)

# Verify missing values are fixed
print("\n✅ Missing values handled successfully!")
print(df_pcos_extended.isnull().sum().sum(), "missing values remaining in PCOS Extended Data")


Missing Values Before Fixing:
Sl. No                    0
Patient File No.          0
PCOS (Y/N)                0
 Age (yrs)                0
Weight (Kg)               0
Height(Cm)                0
BMI                       0
Blood Group               0
Pulse rate(bpm)           0
RR (breaths/min)          0
Hb(g/dl)                  0
Cycle(R/I)                0
Cycle length(days)        0
Marraige Status (Yrs)     3
Pregnant(Y/N)             0
No. of abortions          0
  I   beta-HCG(mIU/mL)    0
II    beta-HCG(mIU/mL)    0
FSH(mIU/mL)               0
LH(mIU/mL)                0
FSH/LH                    0
Hip(inch)                 0
Waist(inch)               0
Waist:Hip Ratio           0
TSH (mIU/L)               0
AMH(ng/mL)                0
PRL(ng/mL)                0
Vit D3 (ng/mL)            0
PRG(ng/mL)                0
RBS(mg/dl)                0
Weight gain(Y/N)          0
hair growth(Y/N)          0
Skin darkening (Y/N)      0
Hair loss(Y/N)            0
Pimples(Y/N)      

In [ ]:
# Select numerical columns (excluding target variable "PCOS (Y/N)")
numerical_cols = df_pcos_extended.select_dtypes(include=['number']).columns.tolist()
numerical_cols.remove("PCOS (Y/N)")

# Apply StandardScaler
scaler = StandardScaler()
df_pcos_extended[numerical_cols] = scaler.fit_transform(df_pcos_extended[numerical_cols])

print("✅ Feature Scaling completed (Z-score normalization applied).")


✅ Feature Scaling completed (Z-score normalization applied).


In [ ]:
# Step 1: Standardize column names (Remove extra spaces & unwanted characters)
df_pcos_extended.columns = (
    df_pcos_extended.columns.str.replace("\s+", " ", regex=True)  # Replace multiple spaces with a single space
                              .str.strip()  # Remove leading and trailing spaces
                              .str.replace("I beta-HCG", "beta-HCG", regex=False)  # Fix incorrect column name
)

# Step 2: Remove non-informative columns (Sl. No, Patient File No.)
df_pcos_extended.drop(columns=["Sl. No", "Patient File No."], errors='ignore', inplace=True)

# Step 3: Convert all numerical values to float
for col in df_pcos_extended.columns:
    df_pcos_extended[col] = pd.to_numeric(df_pcos_extended[col], errors='coerce')

# Step 4: Fill NaN values with the median of each column
df_pcos_extended.fillna(df_pcos_extended.median(numeric_only=True), inplace=True)

# Step 5: Separate features (X) and target variable (y)
X = df_pcos_extended.drop(columns=["PCOS (Y/N)"])
y = df_pcos_extended["PCOS (Y/N)"]

# Step 6: Apply Recursive Feature Elimination (RFE) with RandomForestClassifier
estimator = RandomForestClassifier(random_state=42)
rfe = RFE(estimator, n_features_to_select=20)  # Select top 20 features
X_rfe = rfe.fit_transform(X, y)

# Step 7: Get selected feature names
selected_features = X.columns[rfe.support_]

# Convert back to DataFrame
X_selected = pd.DataFrame(X_rfe, columns=selected_features)

print("✅ Feature Selection completed successfully!")
print("Selected Features:", list(selected_features))


✅ Feature Selection completed successfully!
Selected Features: ['Weight (Kg)', 'Cycle(R/I)', 'Cycle length(days)', 'beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Fast food (Y/N)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)']


In [ ]:
# # Convert 'AMH(ng/mL)' to numeric in both DataFrames
# df_pcos_extended['AMH(ng/mL)'] = pd.to_numeric(df_pcos_extended['AMH(ng/mL)'], errors='coerce')

In [ ]:
# Use the new dataset directly
data = df_pcos_extended.copy()

# Display dataset information
print("✅ Extended PCOS Dataset Loaded!")
print("📊 Dataset Shape:", data.shape)
print(data.head())


✅ Extended PCOS Dataset Loaded!
📊 Dataset Shape: (2000, 42)
   PCOS (Y/N)  Age (yrs)  Weight (Kg)  Height(Cm)       BMI  Blood Group  \
0           0  -0.248511     0.913456    1.868618 -0.265934    -0.975716   
1           0   0.852718     0.365713   -0.390654  0.280117    -0.430014   
2           0   0.852718    -0.671054   -1.217445 -0.086155     0.661392   
3           1  -0.432049     0.643829   -1.288897  0.874030     0.661392   
4           0   0.302104    -0.610965   -0.934857 -0.291594    -0.430014   

   Pulse rate(bpm)  RR (breaths/min)  Hb(g/dl)  Cycle(R/I)  ...  Pimples(Y/N)  \
0        -0.302837          1.590653  0.975064    1.649007  ...      1.047108   
1        -0.790890         -0.707979 -0.753334   -0.602966  ...      1.047108   
2         1.649375          0.441337 -1.329466    1.649007  ...     -0.955011   
3        -0.302837         -0.707979  0.975064    1.649007  ...     -0.955011   
4        -0.302837         -0.707979 -1.099013   -0.602966  ...     -0.955011 

In [ ]:

# Correct the column names to match those in the DataFrame
data = data.drop(columns=["  I   beta-HCG(mIU/mL)", "II    beta-HCG(mIU/mL)"], errors='ignore')
data = data.rename(columns={"PCOS (Y/N)": "Target"})


data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Target                 2000 non-null   int64  
 1   Age (yrs)              2000 non-null   float64
 2   Weight (Kg)            2000 non-null   float64
 3   Height(Cm)             2000 non-null   float64
 4   BMI                    2000 non-null   float64
 5   Blood Group            2000 non-null   float64
 6   Pulse rate(bpm)        2000 non-null   float64
 7   RR (breaths/min)       2000 non-null   float64
 8   Hb(g/dl)               2000 non-null   float64
 9   Cycle(R/I)             2000 non-null   float64
 10  Cycle length(days)     2000 non-null   float64
 11  Marraige Status (Yrs)  2000 non-null   float64
 12  Pregnant(Y/N)          2000 non-null   float64
 13  No. of abortions       2000 non-null   float64
 14  beta-HCG(mIU/mL)       2000 non-null   float64
 15  Ibet

In [ ]:
columns_to_remove = [
    'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Marraige Status (Yrs)',
    'Pregnant(Y/N)', 'No. of abortions', 'Ibeta-HCG(mIU/mL)', 'Hip(inch)',
    'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)',
    'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
    'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)',
    'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)',
    'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)',

]

# Remove the columns from the DataFrame
data = data.drop(columns=columns_to_remove)

# Show the cleaned data
data.head()

,Target,Age (yrs),Weight (Kg),Height(Cm),BMI,Hb(g/dl),Cycle(R/I),Cycle length(days),beta-HCG(mIU/mL),FSH(mIU/mL),LH(mIU/mL),FSH/LH
0,0,-0.248511,0.913456,1.868618,-0.265934,0.975064,1.649007,0.019336,-0.055998,-0.037035,-0.063036,-0.078227
1,0,0.852718,0.365713,-0.390654,0.280117,-0.753334,-0.602966,0.722456,-0.067180,-0.051622,-0.068581,-0.091416
2,0,0.852718,-0.671054,-1.217445,-0.086155,-1.329466,1.649007,-2.090025,-0.193512,-0.051878,-0.070261,-0.088208
3,1,-0.432049,0.643829,-1.288897,0.874030,0.975064,1.649007,0.019336,-0.193512,-0.053107,-0.062700,-0.102288
4,0,0.302104,-0.610965,-0.934857,-0.291594,-1.099013,-0.602966,0.019336,-0.192775,-0.041385,-0.076898,-0.001054


In [ ]:
# Step 1: Check original class distribution
print("\n🔍 Class distribution before SMOTE:")
print(y.value_counts())

# Step 2: Apply SMOTE for perfect class balancing
smote = SMOTE(sampling_strategy='auto', random_state=42)  # Auto ensures full balancing
X_balanced, y_balanced = smote.fit_resample(X_selected, y)

# Step 3: Check new class distribution
print("\n✅ Class imbalance handled using SMOTE.")
print("New dataset shape:", X_balanced.shape)
print("Class distribution after SMOTE:\n", y_balanced.value_counts())



🔍 Class distribution before SMOTE:
PCOS (Y/N)
0    1392
1     608
Name: count, dtype: int64

✅ Class imbalance handled using SMOTE.
New dataset shape: (2784, 20)
Class distribution after SMOTE:
 PCOS (Y/N)
0    1392
1    1392
Name: count, dtype: int64


In [ ]:
# 🔹 Step 1: Fix Data Size Mismatch
min_samples = min(len(X_selected), len(y_balanced))  # Ensure equal number of rows
X_selected = X_selected.iloc[:min_samples].reset_index(drop=True)
y_balanced = y_balanced.iloc[:min_samples].reset_index(drop=True)

print(f"✅ Data Size Fixed! Now X_selected: {X_selected.shape}, y_balanced: {y_balanced.shape}")
print("✅ Clinical Data Ready for MLP Model! 🚀")

✅ Data Size Fixed! Now X_selected: (2000, 20), y_balanced: (2000,)
✅ Clinical Data Ready for MLP Model! 🚀


In [ ]:
data.head()

,Target,Age (yrs),Weight (Kg),Height(Cm),BMI,Hb(g/dl),Cycle(R/I),Cycle length(days),beta-HCG(mIU/mL),FSH(mIU/mL),LH(mIU/mL),FSH/LH
0,0,-0.248511,0.913456,1.868618,-0.265934,0.975064,1.649007,0.019336,-0.055998,-0.037035,-0.063036,-0.078227
1,0,0.852718,0.365713,-0.390654,0.280117,-0.753334,-0.602966,0.722456,-0.067180,-0.051622,-0.068581,-0.091416
2,0,0.852718,-0.671054,-1.217445,-0.086155,-1.329466,1.649007,-2.090025,-0.193512,-0.051878,-0.070261,-0.088208
3,1,-0.432049,0.643829,-1.288897,0.874030,0.975064,1.649007,0.019336,-0.193512,-0.053107,-0.062700,-0.102288
4,0,0.302104,-0.610965,-0.934857,-0.291594,-1.099013,-0.602966,0.019336,-0.192775,-0.041385,-0.076898,-0.001054


In [ ]:
numerical_cols = data.select_dtypes(include=['number']).columns.tolist()
print(numerical_cols)
numerical_cols.remove("Target")  # Don't normalize the target

precomputed_means = data[numerical_cols].mean().to_dict()
precomputed_stds = data[numerical_cols].std().to_dict()

# Save to JSON for later use
import json

save_path = "/content/drive/MyDrive/pcos_detection/scaling_values.json"

# Save to JSON
os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
with open(save_path, "w") as f:
    json.dump({"means": precomputed_means, "stds": precomputed_stds}, f)

print(f"✅ Precomputed values saved at: {save_path}")

['Target', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH']
✅ Precomputed values saved at: /content/drive/MyDrive/pcos_detection/scaling_values.json


CNN PREP

In [ ]:
# dataset_paths = ["/content/drive/My Drive/pcos_detection/1/content/data/enhanced_data/"]  # Change for test set too

# corrupt_images = []
# for paths in dataset_paths:
#   for root, dirs, files in os.walk(paths):
#       for file in files:
#           img_path = os.path.join(root, file)
#           try:
#               img = Image.open(img_path).convert("RGB")
#           except (UnidentifiedImageError, OSError):
#               print(f"⚠️ Corrupt image found: {img_path}")
#               corrupt_images.append(img_path)

#   print(f"Total corrupt images: {len(corrupt_images)}")

#   # 🔹 Optionally delete corrupt images
#   for img_path in corrupt_images:
#       os.remove(img_path)
#       print(f"🗑️ Deleted: {img_path}")


In [ ]:
import os

dataset_paths = ["/content/drive/My Drive/pcos_detection/1/content/data/enhanced_data/"]
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif"}  # Common image formats

def count_images(folder_path):
    image_count = 0
    for dirpath, _, filenames in os.walk(folder_path):
        image_count += sum(1 for f in filenames if os.path.splitext(f)[1].lower() in image_extensions)
    return image_count

for path in dataset_paths:
    num_images = count_images(path)
    print(f"📁 Dataset Path: {path}")
    print(f"🖼️ Number of Images: {num_images}\n")


📁 Dataset Path: /content/drive/My Drive/pcos_detection/1/content/data/enhanced_data/
🖼️ Number of Images: 15392



In [ ]:
import os
from PIL import Image
import numpy as np
import cv2  # OpenCV for image processing
from torch.utils.data import Dataset
from torchvision import transforms
import random

class PCOSImageDataset(Dataset):
    def __init__(self, image_folder, label_sequence, transform=None, augment=True):
        self.image_paths = []
        self.labels = label_sequence
        class_mapping = {"notinfected": 0, "infected": 1}
        class_images = {0: [], 1: []}

        for class_name in os.listdir(image_folder):
            class_path = os.path.join(image_folder, class_name)
            if os.path.isdir(class_path):
                class_label = 0 if class_name == "notinfected" else 1
                image_files = [os.path.join(class_path, img) for img in os.listdir(class_path)
                               if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
                class_images[class_label].extend(sorted(image_files))  # Sort to ensure order

        # Select images in the same order as label_sequence
        class_counters = {0: 0, 1: 0}  # Track usage of images
        for label in label_sequence:
            if class_counters[label] < len(class_images[label]):  # Ensure valid access
                self.image_paths.append(class_images[label][class_counters[label]])
                class_counters[label] += 1  # Move to the next image

        # Data Augmentation transforms (if augment is True)
        self.augment = augment

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),  # Resize images to 224x224
            transforms.ToTensor(),  # Convert images to tensor
        ]) if transform is None else transform  # Use provided transform if any

        # Augmentation transformations
        self.augmentation_transform = transforms.Compose([
            transforms.RandomRotation(degrees=30),  # Random Rotation (-30 to 30 degrees)
            transforms.RandomHorizontalFlip(),  # Random horizontal flip
            transforms.RandomVerticalFlip(),  # Random vertical flip
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # Zooming, crop with random scaling
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),  # Brightness adjustments
        ]) if augment else None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")  # Convert image to RGB

        # Convert to numpy array for processing
        image_np = np.array(image)

        # Apply Watershed Segmentation
        image_segmented = self.apply_watershed(image_np)

        # Apply Multilevel Thresholding to detect cysts
        image_segmented = self.apply_multilevel_thresholding(image_segmented)

        # Convert back to PIL Image
        image_segmented = Image.fromarray(image_segmented)

        # Apply Augmentation if enabled
        if self.augment:
            image_segmented = self.augmentation_transform(image_segmented)  # Apply augmentation

        # Apply basic transformations (resize + tensor conversion)
        image_segmented = self.transform(image_segmented)

        label = self.labels[idx]
        return image_segmented, label

    def apply_watershed(self, image):
        """
        Apply Watershed Segmentation to the image.
        This method detects the follicle boundaries based on intensity changes.
        """
        # Step 1: Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        # Step 2: Apply thresholding to get a binary image (foreground vs background)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Step 3: Remove noise using morphological operations
        kernel = np.ones((3, 3), np.uint8)
        sure_bg = cv2.dilate(thresh, kernel, iterations=3)

        # Step 4: Apply distance transform
        dist_transform = cv2.distanceTransform(thresh, cv2.DIST_L2, 5)
        _, sure_fg = cv2.threshold(dist_transform, 0.7 * dist_transform.max(), 255, 0)

        # Step 5: Subtract sure foreground from sure background to get unknown region
        sure_fg = np.uint8(sure_fg)
        unknown = cv2.subtract(sure_bg, sure_fg)

        # Step 6: Label markers (foreground vs background)
        _, markers = cv2.connectedComponents(sure_fg)

        # Step 7: Apply watershed algorithm
        markers = markers + 1
        markers[unknown == 255] = 0

        # Step 8: Perform watershed algorithm
        image_segmented = image.copy()
        cv2.watershed(image_segmented, markers)

        # Mark boundary pixels
        image_segmented[markers == -1] = [255, 0, 0]  # Red boundary lines

        return image_segmented

    def apply_multilevel_thresholding(self, image):
        """
        Apply Multilevel Thresholding to detect cysts.
        This method identifies cysts by using multiple thresholds.
        """
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        # Use Otsu's method for automatic thresholding, or you can manually choose threshold values
        _, threshold_1 = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        _, threshold_2 = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

        # Combine multiple thresholds to create segmented regions
        segmented_image = np.zeros_like(gray)
        segmented_image[threshold_1 == 255] = 1  # Cyst regions marked as 1
        segmented_image[threshold_2 == 255] = 2  # Follicle regions marked as 2

        # Convert back to RGB for visualization
        segmented_image_rgb = cv2.applyColorMap(segmented_image * 85, cv2.COLORMAP_JET)  # Colorize the output
        return segmented_image_rgb

In [ ]:
class PCOSDataset(Dataset):
    def __init__(self, PCOSImageDataset, data, split="train"):
        # Access the image paths and labels based on the split
        if split == "train":
            self.image_paths = PCOSImageDataset.image_paths
            self.labels = PCOSImageDataset.labels
        else:  # Assume 'test' split if not 'train'
            self.image_paths = PCOSImageDataset.image_paths
            self.labels = PCOSImageDataset.labels

        # Store the tabular data (data) as an attribute
        self.data = data
        self.PCOSImageDataset = PCOSImageDataset  # Store the PCOSImageDataset as an attribute

    def __len__(self):
        # Return the length of the dataset
        return len(self.data)

    def __getitem__(self, idx):
        # Get image and label from the PCOSImageDataset
        image, label = self.PCOSImageDataset[idx]

        # Access the tabular data for the corresponding index
        tabular_features = self.data.iloc[idx].values.astype(np.float32)  # Convert to float32 for neural network compatibility

        # Convert the tabular features to a tensor
        tabular_features = torch.tensor(tabular_features)

        # Return the image, tabular data, and label
        return image, tabular_features, label


In [ ]:
# Assuming 'data' is a DataFrame containing tabular information, including labels
data_path = "/content/drive/My Drive/pcos_detection/1/content/data/enhanced_data/"

# Ensure the dataset has a 'label' column (assuming 'Target' holds the class labels)
data['label'] = data['Target']  # Convert 'Target' column into 'label' (0 or 1)

# Splitting the dataset into training (80%) and testing (20%)
data_train, data_test = train_test_split(data, test_size=0.2, random_state=42, stratify=data['label'])

# Maintain label order for dataset creation
train_label_sequence = data_train['label'].tolist()
test_label_sequence = data_test['label'].tolist()

# Create PCOSImageDataset instances for training and testing
train_image_dataset = PCOSImageDataset(image_folder=data_path, label_sequence=train_label_sequence, augment=True)
test_image_dataset = PCOSImageDataset(image_folder=data_path, label_sequence=test_label_sequence, augment=False)

# Wrap with PCOSDataset to include tabular data
train_dataset = PCOSDataset(train_image_dataset, data_train, split="train")
test_dataset = PCOSDataset(test_image_dataset, data_test, split="test")

print("✅ Dataset successfully split into training and testing sets!")


✅ Dataset successfully split into training and testing sets!


In [ ]:
# ✅ Validate dataset sizes
print(f"📊 Training set images: {len(train_image_dataset)}")
print(f"📊 Testing set images: {len(test_image_dataset)}")

# ✅ Ensure total matches 2000(clicical+ultrsound)
total_images = len(train_image_dataset) + len(test_image_dataset)

📊 Training set images: 1600
📊 Testing set images: 400


In [ ]:
# Convert labels to numpy arrays for easy comparison
train_df_labels = np.array(data_train['label'])  # Labels from DataFrame
train_dataset_labels = np.array(train_image_dataset.labels)  # Labels from dataset

test_df_labels = np.array(data_test['label'])  # Labels from DataFrame
test_dataset_labels = np.array(test_image_dataset.labels)  # Labels from dataset

# Check if both lists are identical for training data
if np.array_equal(train_df_labels, train_dataset_labels):
    print("✅ Labels in data_train match labels in train_image_dataset!")
else:
    print("❌ Labels in data_train do NOT match labels in train_image_dataset!")
    mismatches = np.where(train_df_labels != train_dataset_labels)[0]
    print(f"Found {len(mismatches)} mismatches in training data at indices: {mismatches[:10]}")  # Print first 10 mismatches

# Check if both lists are identical for testing data
if np.array_equal(test_df_labels, test_dataset_labels):
    print("✅ Labels in data_test match labels in test_image_dataset!")
else:
    print("❌ Labels in data_test do NOT match labels in test_image_dataset!")
    mismatches = np.where(test_df_labels != test_dataset_labels)[0]
    print(f"Found {len(mismatches)} mismatches in testing data at indices: {mismatches[:10]}")  # Print first 10 mismatches

✅ Labels in data_train match labels in train_image_dataset!
✅ Labels in data_test match labels in test_image_dataset!


In [ ]:
import torch
from torch.utils.data import DataLoader

# Drop the 'Target' column as it has been assigned to 'label'
data_train = data_train.drop(columns=['Target'])
data_test = data_test.drop(columns=['Target'])
data_train = data_train.drop(columns=['label'])
data_test = data_test.drop(columns=['label'])

print(data_train.head())
print(data_test.head())

# Create dataset instances
train_dataset = PCOSDataset(train_image_dataset, data_train, split="train")
test_dataset = PCOSDataset(test_image_dataset, data_test, split="test")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print("✅ DataLoaders created successfully!")


      Age (yrs)  Weight (Kg)  Height(Cm)       BMI  Hb(g/dl)  Cycle(R/I)  \
971    0.118566     0.649315    0.610901 -0.024517 -1.099013   -0.602966   
1396  -1.349740    -0.212098    2.173316 -0.858094 -1.329466   -0.602966   
387    0.302104     0.117491    0.822986 -0.587432  0.398931    1.649007   
916    0.118566    -1.173181   -0.927646 -1.041313  0.398931   -0.602966   
1937  -1.349740    -0.358582   -0.268853 -0.439937 -0.753334   -0.602966   

      Cycle length(days)  beta-HCG(mIU/mL)  FSH(mIU/mL)  LH(mIU/mL)    FSH/LH  
971             0.722456         -0.192239    -0.055103   -0.065976 -0.102109  
1396            0.019336         -0.193512    -0.036779   -0.020192 -0.107456  
387             0.019336         -0.192794    -0.046555   -0.072697 -0.065038  
916             0.722456         -0.193512    -0.041692   -0.045142 -0.102288  
1937            0.019336         -0.184488    -0.026798   -0.054887 -0.080009  
      Age (yrs)  Weight (Kg)  Height(Cm)       BMI  Hb(g/dl)  C

In [ ]:
# Check the shape of one batch from the training DataLoader
for batch in train_loader:
    images, tabular_data, labels = batch
    print(f"🖼️ Images shape: {images.shape}")  # (batch_size, channels, height, width)
    print(f"📊 Tabular data shape: {tabular_data.shape}")  # (batch_size, num_features)
    print(f"🏷️ Labels shape: {labels.shape}")  # (batch_size,)
    break  # Print only the first batch

# Check the shape of one batch from the test DataLoader
for batch in test_loader:
    images, tabular_data, labels = batch
    print(f"🖼️ Images shape: {images.shape}")  # (batch_size, channels, height, width)
    print(f"📊 Tabular data shape: {tabular_data.shape}")  # (batch_size, num_features)
    print(f"🏷️ Labels shape: {labels.shape}")  # (batch_size,)
    break  # Print only the first batch


🖼️ Images shape: torch.Size([32, 3, 224, 224])
📊 Tabular data shape: torch.Size([32, 11])
🏷️ Labels shape: torch.Size([32])
🖼️ Images shape: torch.Size([32, 3, 224, 224])
📊 Tabular data shape: torch.Size([32, 11])
🏷️ Labels shape: torch.Size([32])


In [ ]:
# Count total images in train and test datasets
num_train_images = len(train_image_dataset.image_paths)
num_test_images = len(test_image_dataset.image_paths)

print(f"📂 Total training images: {num_train_images}")
print(f"📂 Total testing images: {num_test_images}")

# Optional: Count images in each class
train_class_counts = {0: 0, 1: 0}
test_class_counts = {0: 0, 1: 0}

for label in train_image_dataset.labels:
    train_class_counts[label] += 1

for label in test_image_dataset.labels:
    test_class_counts[label] += 1

print(f"🔹 Training class distribution: {train_class_counts}")
print(f"🔹 Testing class distribution: {test_class_counts}")


📂 Total training images: 1600
📂 Total testing images: 400
🔹 Training class distribution: {0: 1114, 1: 486}
🔹 Testing class distribution: {0: 278, 1: 122}


**Model Building**

---

info on model

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class PCOS_MLP(nn.Module):
    def __init__(self, input_dim):
        super(PCOS_MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 32),  # Output Layer for tabular data
            nn.Sigmoid()  # Probability Output for binary classification
        )

    def forward(self, x):
        return self.mlp(x)  # Expected output shape: [batch_size, 1]

class PCOS_CNN(nn.Module):
    def __init__(self, num_classes=2):
        super(PCOS_CNN, self).__init__()
        self.cnn = models.resnet18(pretrained=True)  # Use ResNet18 as the backbone
        self.cnn.fc = nn.Linear(512, 32)  # Modify the last layer to output a 32D feature vector

        self.classifier = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 512)  # Final output layer for classification
        )

    def forward(self, x):
        x = self.cnn(x)  # Extract image features (Shape: [batch_size, 32])
        return x  # Returning 32D features


class PCOS_Multimodal(nn.Module):
    def __init__(self, tabular_input_dim, num_classes=2):
        super(PCOS_Multimodal, self).__init__()
        self.mlp_model = PCOS_MLP(tabular_input_dim)
        self.cnn_model = PCOS_CNN(num_classes)

        # Final classifier to combine both the outputs
        self.fc = nn.Sequential(
            nn.Linear(32 + 32, 128),  # 1 from MLP output and 32 from CNN output
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),  # Final output layer for classification
            nn.Softmax(dim=1)  # Output class probabilities
        )

    def forward(self, tabular_input, image_input):
        # Process the tabular data through the MLP model
        mlp_output = self.mlp_model(tabular_input)  # Shape: [batch_size, 1]
        # print(f"MLP Output Shape: {mlp_output.shape}")

        # Process the image data through the CNN model
        cnn_output = self.cnn_model(image_input)  # Shape: [batch_size, num_classes]
        # print(f"CNN Output Shape: {cnn_output.shape}")

        # Concatenate the outputs of both models (MLP and CNN)
        combined_input = torch.cat((mlp_output, cnn_output), dim=1)  # Concatenate along the feature dimension
        # print(f"Combined Input Shape: {combined_input.shape}")

        # Pass through the final classifier
        output = self.fc(combined_input)
        return output



In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score


# Assuming your tabular data is a pandas DataFrame
tabular_input_dim = tabular_data.shape[1]
print(tabular_input_dim)
model = PCOS_Multimodal(tabular_input_dim=tabular_input_dim, num_classes=2)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # For classification tasks
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 10  # Adjust based on your needs
for epoch in range(epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for image_data,tabular_data, labels in train_loader:
        # Move data to device (GPU or CPU)
        tabular_data = tabular_data.to(device)
        image_data = image_data.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(tabular_data, image_data)

        # Calculate the loss
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Print statistics for each epoch
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")

# Testing Loop
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
correct = 0
total = 0
all_predictions = []
all_labels = []

with torch.no_grad():  # No gradients needed for testing
    for image_data, tabular_data, labels in test_loader:
        tabular_data = tabular_data.to(device)
        image_data = image_data.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(tabular_data, image_data)

        # Calculate the loss
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Store predictions for later evaluation
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print the final statistics
test_loss /= len(test_loader)
test_accuracy = 100 * correct / total
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")


11
Epoch 1/10, Loss: 0.3979, Accuracy: 92.19%
Epoch 2/10, Loss: 0.3533, Accuracy: 96.25%
Epoch 3/10, Loss: 0.3613, Accuracy: 95.31%
Epoch 4/10, Loss: 0.3518, Accuracy: 96.19%
Epoch 5/10, Loss: 0.3570, Accuracy: 95.69%
Epoch 6/10, Loss: 0.3563, Accuracy: 95.88%
Epoch 7/10, Loss: 0.3364, Accuracy: 97.62%
Epoch 8/10, Loss: 0.3455, Accuracy: 96.81%
Epoch 9/10, Loss: 0.3289, Accuracy: 98.44%
Epoch 10/10, Loss: 0.3308, Accuracy: 98.44%
Test Loss: 0.3172, Test Accuracy: 99.75%


In [ ]:
import os
import torch

# Define the directory to save the model
save_directory = "/content/drive/MyDrive/pcos_detection/"
os.makedirs(save_directory, exist_ok=True)  # Creates the directory if it doesn't exist

# Define the model path
model_path = os.path.join(save_directory, "multimodal_model_final.pth")

# Save the model state_dict
torch.save(model.state_dict(), model_path)

print(f"Model saved at: {model_path}")

Model saved at: /content/drive/MyDrive/pcos_detection/multimodal_model_final.pth
